In [1]:
import os
import io
import boto3
import tarfile
import pgzip
import concurrent.futures
from botocore.config import Config
from smart_open import open

s3 = boto3.client('s3')
bucket_name = 'airborne-smce-prod-user-bucket'
prefix = 'JOIN/results/swe/benchmark/'  # The folder in S3 you want to archive
output_key = f's3://{bucket_name}JOIN/results/swe/swe_2018-2019_season.tar.gz'

# Automatically use all available CPU cores for compression

cpu_cores = os.cpu_count() or 4
MAX_CONCURRENT_DOWNLOADS = 20  # Adjust based on your available RAM and file sizes

def download_s3_file(bucket, key):
    """Worker function: Downloads a single file from S3 entirely into RAM."""
    response = s3.get_object(Bucket=bucket, Key=key)
    # .read() executes the actual download in the background thread
    data = response['Body'].read()
    return key, data

print(f"Starting multi-threaded compression ({cpu_cores} cores) with concurrent I/O...")

with open(output_key, 'wb') as fout:
    with pgzip.PgzipFile(fileobj=fout, mode='w', thread=cpu_cores) as pgz:
        with tarfile.open(fileobj=pgz, mode='w|') as tar:
            
            paginator = s3.get_paginator('list_objects_v2')
            
            # 2. Spin up a thread pool for concurrent downloading
            with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_CONCURRENT_DOWNLOADS) as executor:
                futures = set()
                
                for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
                    for obj in page.get('Contents', []):
                        key = obj['Key']
                        
                        if key.endswith('/'):
                            continue
                            
                        # Submit the download task to the background threads
                        future = executor.submit(download_s3_file, bucket_name, key)
                        futures.add(future)
                        
                        # 3. If we hit our concurrency limit, stop submitting and wait for downloads to finish.
                        # This prevents us from downloading the entire bucket into RAM all at once.
                        while len(futures) >= MAX_CONCURRENT_DOWNLOADS:
                            
                            # Wait until AT LEAST ONE background download finishes
                            done, futures = concurrent.futures.wait(
                                futures, 
                                return_when=concurrent.futures.FIRST_COMPLETED
                            )
                            
                            # Write the completed downloads to the tar stream
                            for f in done:
                                k, data = f.result()
                                tar_info = tarfile.TarInfo(name=k)
                                tar_info.size = len(data)
                                
                                # Wrap the byte data in BytesIO so tarfile can read it like a file
                                tar.addfile(tarinfo=tar_info, fileobj=io.BytesIO(data))
                
                # 4. We finished looping through S3. Drain whatever downloads are left in the queue.
                for f in concurrent.futures.as_completed(futures):
                    k, data = f.result()
                    tar_info = tarfile.TarInfo(name=k)
                    tar_info.size = len(data)
                    tar.addfile(tarinfo=tar_info, fileobj=io.BytesIO(data))

print("Archive complete! CPUs should have been working hard.")

Starting multi-threaded compression (8 cores) with concurrent I/O...
Archive complete! CPUs should have been working hard.
